In [ ]:
import uuid
import time
import numpy as np
import scipy.sparse as sp
from pymilvus import MilvusClient, DataType, AnnSearchRequest, RRFRanker

In [ ]:
# ════════════════════════════════════════════════════════════
# 1. CONNECTION & INITIALIZATION
# ════════════════════════════════════════════════════════════

# ── 1a. Local file-based (Milvus Lite) ──────────────────────
# Data persists across sessions in the .db file
client = MilvusClient("./milvus_learn.db")
print("✅ Connected to local Milvus Lite file.")

# ── 1b. Remote Milvus server (Docker / Kubernetes) ──────────
# client = MilvusClient(uri="http://localhost:19530")

# ── 1c. Connectivity check ──────────────────────────────────
def check_connection(client: MilvusClient) -> bool:
    try:
        client.list_collections()
        print("✅ Connection healthy.")
        return True
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return False

check_connection(client)

In [ ]:
# ════════════════════════════════════════════════════════════
# 2. COLLECTION MANAGEMENT
# ════════════════════════════════════════════════════════════

COLLECTION_NAME = "learn_milvus"
VECTOR_DIM      = 8   # small dim for demo; use 1536 for OpenAI text-embedding-3-small

# ── 2a. Check existence ─────────────────────────────────────
def collection_exists(client, name: str) -> bool:
    return client.has_collection(name)

print(f"\nCollection exists: {collection_exists(client, COLLECTION_NAME)}")

# ── 2b. Drop if exists (clean slate) ────────────────────────
if collection_exists(client, COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)
    print(f"🗑️  Dropped existing collection '{COLLECTION_NAME}'.")

# ── 2c. View details of an existing collection ──────────────
def describe_collection(client, name: str):
    if not collection_exists(client, name):
        print(f"Collection '{name}' does not exist.")
        return
    info = client.describe_collection(name)
    print(f"\n── Collection: {name} ──")
    print(f"  Fields     : {[f['name'] for f in info['fields']]}")
    print(f"  Load State : {client.get_load_state(name)}")

# ── 2d. Drop collection ─────────────────────────────────────
def safe_drop(client, name: str):
    if collection_exists(client, name):
        client.drop_collection(name)
        print(f"🗑️  Dropped '{name}'.")
    else:
        print(f"'{name}' does not exist, nothing to drop.")

In [ ]:
# ════════════════════════════════════════════════════════════
# 3. SCHEMA DESIGN & INDEX CREATION
# ════════════════════════════════════════════════════════════

# ── 3a. Full schema design example ──────────────────────────
schema = client.create_schema(
    auto_id=False,          # we supply our own IDs
    enable_dynamic_field=True   # 3b. allows storing extra keys not in schema
)

# Primary key
schema.add_field("id",          DataType.VARCHAR,       is_primary=True, max_length=100)

# Dense vector field
schema.add_field("vector",      DataType.FLOAT_VECTOR,  dim=VECTOR_DIM)

# Sparse vector field (for BM25 / hybrid search) — no dim needed
schema.add_field("sparse_vector", DataType.SPARSE_FLOAT_VECTOR)

# Scalar metadata fields
schema.add_field("text",        DataType.VARCHAR,       max_length=65535)
schema.add_field("page_number", DataType.INT64)
schema.add_field("source",      DataType.VARCHAR,       max_length=500)
schema.add_field("chunk_id",    DataType.INT64)
# Extra field for demonstrating upsert / delete
schema.add_field("category",    DataType.VARCHAR,       max_length=100)

# ── 3c. Index creation ──────────────────────────────────────
index_params = client.prepare_index_params()

# Dense index — FLAT is exact (good for dev); HNSW is ANN (good for prod)
index_params.add_index(
    field_name="vector",
    index_type="FLAT",          # swap to "HNSW" for large-scale
    metric_type="COSINE"        # COSINE | IP | L2
)

# Sparse index — mandatory for sparse fields
index_params.add_index(
    field_name="sparse_vector",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="IP"            # Inner Product is the only valid metric for sparse
)

# Create the collection with schema + indexes in one call
client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
    index_params=index_params
)
print(f"\n✅ Collection '{COLLECTION_NAME}' created with schema and indexes.")

# Describe it
describe_collection(client, COLLECTION_NAME)


In [ ]:
# ════════════════════════════════════════════════════════════
# 4. DATA OPERATIONS
# ════════════════════════════════════════════════════════════

# Helper — generate fake sparse dict mimicking BM25 output
def fake_sparse(vocab_size=100, nnz=5) -> dict:
    indices = np.random.choice(vocab_size, nnz, replace=False)
    values  = np.random.rand(nnz).astype(float)
    return {int(i): float(v) for i, v in zip(indices, values)}

# Helper — generate fake dense vector
def fake_dense(dim=VECTOR_DIM) -> list:
    v = np.random.rand(dim).astype(float)
    return (v / np.linalg.norm(v)).tolist()   # normalize for COSINE

# ── 4a. INSERT entities ─────────────────────────────────────
# Insert always adds new rows. Duplicate primary key = error.
insert_data = [
    {
        "id"           : str(uuid.uuid4()),
        "vector"       : fake_dense(),
        "sparse_vector": fake_sparse(),
        "text"         : "Employees are entitled to 18 days of annual leave per year.",
        "page_number"  : 1,
        "source"       : "hr_policy.pdf",
        "chunk_id"     : 0,
        "category"     : "leave",
        # dynamic field — not in schema, stored automatically because enable_dynamic_field=True
        "extra_note"   : "This is a dynamic field — no schema change needed."
    },
    {
        "id"           : str(uuid.uuid4()),
        "vector"       : fake_dense(),
        "sparse_vector": fake_sparse(),
        "text"         : "Sick leave can be taken up to 7 days without a medical certificate.",
        "page_number"  : 2,
        "source"       : "hr_policy.pdf",
        "chunk_id"     : 1,
        "category"     : "leave"
    },
    {
        "id"           : str(uuid.uuid4()),
        "vector"       : fake_dense(),
        "sparse_vector": fake_sparse(),
        "text"         : "Performance appraisals are conducted bi-annually in April and October.",
        "page_number"  : 5,
        "source"       : "hr_policy.pdf",
        "chunk_id"     : 2,
        "category"     : "performance"
    },
    {
        "id"           : "fixed-id-for-upsert-demo",   # fixed ID so we can upsert it later
        "vector"       : fake_dense(),
        "sparse_vector": fake_sparse(),
        "text"         : "Original text — will be overwritten by upsert.",
        "page_number"  : 10,
        "source"       : "hr_policy.pdf",
        "chunk_id"     : 3,
        "category"     : "other"
    },
]

res = client.insert(collection_name=COLLECTION_NAME, data=insert_data)
print(f"\n✅ Inserted {res['insert_count']} records.")

# ── Load into memory before any search / query ──────────────
client.load_collection(COLLECTION_NAME)
print(f"✅ Collection loaded. State: {client.get_load_state(COLLECTION_NAME)}")

# ── 4b. UPSERT entities ─────────────────────────────────────
# Upsert: updates the row if primary key exists, inserts if it doesn't.
upsert_data = [
    {
        "id"           : "fixed-id-for-upsert-demo",   # same ID as above → will UPDATE
        "vector"       : fake_dense(),
        "sparse_vector": fake_sparse(),
        "text"         : "UPSERTED text — this replaced the original row.",
        "page_number"  : 10,
        "source"       : "hr_policy.pdf",
        "chunk_id"     : 3,
        "category"     : "upserted"
    }
]

res = client.upsert(collection_name=COLLECTION_NAME, data=upsert_data)
print(f"\n✅ Upserted {res['upsert_count']} record(s).")

# Verify the upsert worked
upserted_row = client.query(
    collection_name=COLLECTION_NAME,
    filter='id == "fixed-id-for-upsert-demo"',
    output_fields=["text", "category"]
)
print(f"   Upserted row text: {upserted_row[0]['text']}")
print(f"   Upserted category: {upserted_row[0]['category']}")

# ── 4c. DELETE entities ─────────────────────────────────────
# Method 1: delete by primary key list
ids_to_delete = [insert_data[2]["id"]]   # delete the performance chunk
client.delete(collection_name=COLLECTION_NAME, ids=ids_to_delete)
print(f"\n✅ Deleted entity with id: {ids_to_delete[0][:8]}...")

# Method 2: delete by filter expression
client.delete(
    collection_name=COLLECTION_NAME,
    filter='category == "upserted"'      # deletes all rows where category == "upserted"
)
print("✅ Deleted all entities where category == 'upserted'.")

# Confirm remaining count
remaining = client.query(
    collection_name=COLLECTION_NAME,
    filter="",
    output_fields=["id", "category"]
)
print(f"   Remaining rows: {len(remaining)}")

In [ ]:
# ════════════════════════════════════════════════════════════
# 5. SEARCH & RETRIEVAL
# ════════════════════════════════════════════════════════════

# Re-insert some data so searches return results
fresh_data = [
    {
        "id"           : str(uuid.uuid4()),
        "vector"       : fake_dense(),
        "sparse_vector": fake_sparse(),
        "text"         : "Annual leave policy grants 18 days per calendar year.",
        "page_number"  : 1,
        "source"       : "hr_policy.pdf",
        "chunk_id"     : 10,
        "category"     : "leave"
    },
    {
        "id"           : str(uuid.uuid4()),
        "vector"       : fake_dense(),
        "sparse_vector": fake_sparse(),
        "text"         : "Sick leave entitlement is 7 days without medical proof.",
        "page_number"  : 2,
        "source"       : "hr_policy.pdf",
        "chunk_id"     : 11,
        "category"     : "leave"
    },
    {
        "id"           : str(uuid.uuid4()),
        "vector"       : fake_dense(),
        "sparse_vector": fake_sparse(),
        "text"         : "Salary increment is tied to the bi-annual performance review.",
        "page_number"  : 8,
        "source"       : "hr_policy.pdf",
        "chunk_id"     : 12,
        "category"     : "performance"
    },
]

client.insert(collection_name=COLLECTION_NAME, data=fresh_data)
# After insert, reload to reflect new data
client.load_collection(COLLECTION_NAME)

query_vector = fake_dense()   # in real code: embedding_obj.embed_query(query)

# ── 5a. VECTOR SEARCH (dense only) ──────────────────────────
print("\n── 5a. Dense Vector Search ──")
results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    anns_field="vector",
    param={"metric_type": "COSINE"},
    limit=3,
    output_fields=["text", "page_number", "category"]
)
for hit in results[0]:
    print(f"  Score: {hit['distance']:.4f} | Page: {hit['entity']['page_number']} | {hit['entity']['text'][:60]}")

# ── 5b. FILTERING (scalar pre-filter) ───────────────────────
# Filter runs BEFORE vector search — not post-filtering.
# Supported operators: ==, !=, >, >=, <, <=, in, not in, like, &&, ||
print("\n── 5b. Filtered Vector Search (only 'leave' category, page <= 3) ──")
results_filtered = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    anns_field="vector",
    param={"metric_type": "COSINE"},
    filter='category == "leave" && page_number <= 3',   # scalar pre-filter
    limit=3,
    output_fields=["text", "page_number", "category"]
)
for hit in results_filtered[0]:
    print(f"  Score: {hit['distance']:.4f} | Page: {hit['entity']['page_number']} | Cat: {hit['entity']['category']}")

# ── 5c. HYBRID SEARCH (dense + sparse) ──────────────────────
print("\n── 5c. Hybrid Search (Dense + Sparse with RRF) ──")
query_sparse_dict = fake_sparse()   # in real code: sparse_to_dict(bm25_ef.encode_queries([query])[0])

dense_req = AnnSearchRequest(
    data=[query_vector],
    anns_field="vector",
    param={"metric_type": "COSINE"},
    limit=3
)
sparse_req = AnnSearchRequest(
    data=[query_sparse_dict],
    anns_field="sparse_vector",
    param={"metric_type": "IP"},
    limit=3
)

hybrid_results = client.hybrid_search(
    collection_name=COLLECTION_NAME,
    reqs=[dense_req, sparse_req],
    ranker=RRFRanker(k=60),
    limit=3,
    output_fields=["text", "page_number", "category"]
)
for hit in hybrid_results[0]:
    print(f"  RRF Score: {hit['distance']:.4f} | Page: {hit['entity']['page_number']} | {hit['entity']['text'][:60]}")

In [ ]:
# ════════════════════════════════════════════════════════════
# 6. ADVANCED TOPICS (often missing from tutorials)
# ════════════════════════════════════════════════════════════

# ── 6a. LOAD STATE MANAGEMENT ───────────────────────────────
def ensure_loaded(client, name: str):
    """Call before every search/query block."""
    state = client.get_load_state(name)
    if state["state"] != "Loaded":
        print(f"  Loading '{name}'...")
        client.load_collection(name)
    else:
        print(f"  '{name}' already loaded.")

ensure_loaded(client, COLLECTION_NAME)

# ── 6b. CONSISTENCY LEVELS ──────────────────────────────────
# Controls when inserted data becomes visible to searches.
# "Strong"   — read-your-writes; slight latency cost
# "Eventual" — fastest; newly inserted data may not appear immediately
# "Session"  — strong within the same session only (default for Lite)
print("\n── 6b. Search with consistency level ──")
results_strong = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    anns_field="vector",
    param={"metric_type": "COSINE"},
    limit=2,
    consistency_level="Strong",        # guarantees freshest data
    output_fields=["text"]
)
print(f"  Strong-consistency results: {len(results_strong[0])}")

# ── 6c. PARTITIONING ────────────────────────────────────────
# Partitions are sub-segments of a collection.
# Use case: isolate data by department, date, user_id, etc.
PART_COLLECTION = "learn_partitions"
if collection_exists(client, PART_COLLECTION):
    client.drop_collection(PART_COLLECTION)

part_schema = client.create_schema()
part_schema.add_field("id",     DataType.VARCHAR, is_primary=True, max_length=100)
part_schema.add_field("vector", DataType.FLOAT_VECTOR, dim=VECTOR_DIM)
part_schema.add_field("text",   DataType.VARCHAR, max_length=1000)

part_index = client.prepare_index_params()
part_index.add_index("vector", index_type="FLAT", metric_type="COSINE")

client.create_collection(PART_COLLECTION, schema=part_schema, index_params=part_index)

# Create partitions
client.create_partition(PART_COLLECTION, partition_name="hr_department")
client.create_partition(PART_COLLECTION, partition_name="finance_department")
print(f"\n✅ Partitions created: {client.list_partitions(PART_COLLECTION)}")

# Insert into a specific partition
client.insert(
    collection_name=PART_COLLECTION,
    data=[{"id": str(uuid.uuid4()), "vector": fake_dense(), "text": "HR leave policy doc."}],
    partition_name="hr_department"
)
client.insert(
    collection_name=PART_COLLECTION,
    data=[{"id": str(uuid.uuid4()), "vector": fake_dense(), "text": "Finance expense policy."}],
    partition_name="finance_department"
)
client.load_collection(PART_COLLECTION)

# Search ONLY within one partition — faster for large collections
results_part = client.search(
    collection_name=PART_COLLECTION,
    data=[fake_dense()],
    anns_field="vector",
    param={"metric_type": "COSINE"},
    limit=2,
    partition_names=["hr_department"],   # scoped to this partition only
    output_fields=["text"]
)
print(f"  Partition search (hr_department): {results_part[0][0]['entity']['text']}")

# ── 6d. SCHEMA EVOLUTION ────────────────────────────────────
# Add a new field to an EXISTING live collection — no data loss, no rebuild.
# New field is nullable by default (existing rows get null).
print("\n── 6d. Schema Evolution — adding a field to live collection ──")
client.add_collection_field(
    collection_name=COLLECTION_NAME,
    field_name="department",
    datatype=DataType.VARCHAR,
    max_length=200,
    nullable=True           # existing rows will have null for this field
)
print("  ✅ Added 'department' field to existing collection without data loss.")

# Verify
describe_collection(client, COLLECTION_NAME)

# ── 6e. BACKUP & MIGRATION PATH ─────────────────────────────
# Milvus Lite stores everything in a single .db file.
# To back up: just copy the file.
# To migrate to full Milvus (Docker): use pymilvus bulk_writer or milvus-backup tool.
print("\n── 6e. Backup path ──")
print("  Milvus Lite DB file: ./milvus_learn.db")
print("  To back up: cp ./milvus_learn.db ./milvus_learn_backup.db")
print("  To migrate to Docker Milvus: use `milvus-backup` CLI tool or bulk_writer API.")

In [ ]:
## QUICK REFERENCE CHEATSHEET

┌─────────────────────────────────────────────────────────────────┐
│                   MILVUS QUICK REFERENCE                        │
├──────────────────────────┬──────────────────────────────────────┤
│ Operation                │ Code                                 │
├──────────────────────────┼──────────────────────────────────────┤
│ Connect (Lite)           │ MilvusClient("./file.db")            │
│ Connect (Server)         │ MilvusClient("http://host:19530")    │
│ List collections         │ client.list_collections()            │
│ Check exists             │ client.has_collection(name)          │
│ Create collection        │ client.create_collection(...)        │
│ Describe collection      │ client.describe_collection(name)     │
│ Drop collection          │ client.drop_collection(name)         │
│ Load to memory           │ client.load_collection(name)         │
│ Get load state           │ client.get_load_state(name)          │
│ Insert                   │ client.insert(name, data)            │
│ Upsert                   │ client.upsert(name, data)            │
│ Delete by ID             │ client.delete(name, ids=[...])       │
│ Delete by filter         │ client.delete(name, filter="...")    │
│ Vector search            │ client.search(name, data, ...)       │
│ Hybrid search            │ client.hybrid_search(name, reqs, .)  │
│ Scalar query             │ client.query(name, filter="...")      │
│ Create partition         │ client.create_partition(name, pname) │
│ Search in partition      │ search(..., partition_names=[...])   │
│ Add field (live)         │ client.add_collection_field(...)     │
├──────────────────────────┼──────────────────────────────────────┤
│ Index types              │ FLAT (exact) | HNSW (ANN, fast)      │
│ Dense metrics            │ COSINE | IP | L2                     │
│ Sparse metric            │ IP only                              │
│ Consistency levels       │ Strong | Session | Eventual          │
└──────────────────────────┴──────────────────────────────────────┘
